# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Coder-bot1/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys
import pandas as pd
import numpy as np

# Load starter data locally or via HF
if os.path.exists("data/raw/content_refresh_anonymized.csv"):
    DATA_PATH = "data/raw/content_refresh_anonymized.csv"
else:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")
    from huggingface_hub import hf_hub_download
    DATA_PATH = hf_hub_download(
        repo_id="FlyRank/internship-starter",
        filename="content_refresh_anonymized.csv",
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Dataset path:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist()[:10], "... total:", len(df.columns))


Dataset path: data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count'] ... total: 44


In [2]:
# Convert fields used by the baseline to numeric values
for col in [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# The dataset already provides CTR.
# Use the observed 90-day CTR rather than a future outcome.
df["observed_ctr"] = df["ctr"]

print("Baseline fields prepared.")

display(
    df[
        [
            "content_id",
            "search_volume",
            "impressions_90d",
            "clicks_90d",
            "observed_ctr",
            "avg_position"
        ]
    ].head()
)

Baseline fields prepared.


,content_id,search_volume,impressions_90d,clicks_90d,observed_ctr,avg_position
0,content_304f48230142,10.0,3803,29,0.76,10.6
1,content_a1fb4e703a9e,90.0,15320,7,0.05,20.3
2,content_9aa793d4d895,0.0,12581,11,0.09,36.5
3,content_331d6c4de07b,10.0,11751,58,0.49,6.2
4,content_d99b7a2d90ca,0.0,19140,24,0.13,44.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal checks

Before building the baseline, I checked two signals that are relevant to FlyRank's prioritization logic.

**Signal 1 — Search volume:** FALSE.

The observed search-volume buckets did not show a consistent increase in observed impressions or clicks from low to high volume. Therefore, I do not use search volume as a scoring signal in this baseline.

**Signal 2 — CTR versus average position:** CONFIRMED.

The observed CTR decreases consistently as average position becomes worse:

- Positions 1–5: average CTR = 1.5729%
- Positions 6–10: average CTR = 0.5117%
- Positions 11+: average CTR = 0.2629%

The pattern supports the CTR-fix hypothesis directionally. I therefore use this confirmed signal for the baseline.

### Baseline rule

I prioritize content that has measurable search visibility and relatively low observed CTR.

A content item receives:

- **1 point** if `avg_position > 0` and its observed CTR is at or below the 25th percentile among items with measurable position.
- **0 points** otherwise.

### Reason codes

- `CTR_FIX` — measurable search visibility with relatively low observed CTR.
- `NO_PRIORITY_SIGNAL` — does not meet the baseline condition.

### Action labels

- `CTR_FIX` — prioritize the item for review of search-result presentation.
- `NO_ACTION` — no action recommended by this baseline.

This is a hand-written, transparent decision-support rule. It is not a fitted model and is not a prediction of future performance.

In [3]:
# ---------------------------------------------------------
# SIGNAL 1 — SEARCH VOLUME
# ---------------------------------------------------------

volume_data = df.dropna(subset=["search_volume"]).copy()

# Rank first so ties do not cause qcut to fail.
volume_data["volume_bucket"] = pd.qcut(
    volume_data["search_volume"].rank(method="first"),
    q=3,
    labels=["Low", "Medium", "High"]
)

volume_table = (
    volume_data
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        avg_search_volume=("search_volume", "mean"),
        avg_impressions_90d=("impressions_90d", "mean"),
        avg_clicks_90d=("clicks_90d", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — SEARCH VOLUME")
display(volume_table)

# ---------------------------------------------------------
# SIGNAL 2 — CTR VS AVERAGE POSITION
# ---------------------------------------------------------

position_data = df[
    (df["avg_position"] > 0) &
    (df["observed_ctr"].notna())
].copy()

position_data["position_bucket"] = pd.cut(
    position_data["avg_position"],
    bins=[0, 5, 10, np.inf],
    labels=[
        "Positions 1-5",
        "Positions 6-10",
        "Positions 11+"
    ],
    include_lowest=True
)

position_table = (
    position_data
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        avg_position=("avg_position", "mean"),
        avg_ctr=("observed_ctr", "mean"),
        avg_impressions_90d=("impressions_90d", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS AVERAGE POSITION")
display(position_table)

print("\nRecorded signal verdicts:")
print("Search volume: FALSE")
print("CTR vs average position: CONFIRMED")

SIGNAL 1 — SEARCH VOLUME


,volume_bucket,n,avg_search_volume,avg_impressions_90d,avg_clicks_90d
0,Low,9178,0.000000,5938.806712,18.870560
1,Medium,9177,7.926338,5149.377574,17.105699
2,High,9177,468.738150,5776.621227,16.143075



SIGNAL 2 — CTR VS AVERAGE POSITION


,position_bucket,n,avg_position,avg_ctr,avg_impressions_90d
0,Positions 1-5,3923,3.599006,1.572937,9753.568952
1,Positions 6-10,9060,7.330419,0.511708,6474.484768
2,Positions 11+,15812,25.913161,0.262900,3736.822350



Recorded signal verdicts:
Search volume: FALSE
CTR vs average position: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline uses the confirmed CTR-versus-position signal.

The low-CTR threshold is the 25th percentile of observed CTR among content items with measurable search position.

The rule is:

- score = 1 when `avg_position > 0` and observed CTR is at or below the threshold;
- score = 0 otherwise.

Each row receives exactly one reason code and one action label.

The queue is ranked by score, then by lower observed CTR, then by higher observed impressions.

No product flag, label-derived field, or future-window outcome is used as an input to the rule.

In [4]:
# ---------------------------------------------------------
# CALCULATE LOW-CTR THRESHOLD
# ---------------------------------------------------------

visible_ctr = df.loc[
    (df["avg_position"] > 0) &
    (df["observed_ctr"].notna()),
    "observed_ctr"
]

ctr_threshold = visible_ctr.quantile(0.25)

print("Low-CTR threshold:", ctr_threshold)
print("Rows with measurable position:", len(visible_ctr))

# ---------------------------------------------------------
# POSITION-ADJUSTED CTR BASELINE
# ---------------------------------------------------------

baseline = df.copy()

# Make sure the fields are numeric
baseline["avg_position"] = pd.to_numeric(
    baseline["avg_position"],
    errors="coerce"
)

baseline["observed_ctr"] = pd.to_numeric(
    baseline["observed_ctr"],
    errors="coerce"
)

# ---------------------------------------------------------
# CREATE POSITION BUCKETS
# ---------------------------------------------------------

baseline["position_bucket"] = pd.cut(
    baseline["avg_position"],
    bins=[0, 5, 10, np.inf],
    labels=[
        "Positions 1-5",
        "Positions 6-10",
        "Positions 11+"
    ],
    include_lowest=True
)

# Only rows with usable position and CTR
baseline["has_position"] = (
    baseline["avg_position"] > 0
) & (
    baseline["observed_ctr"].notna()
)

# ---------------------------------------------------------
# MEDIAN CTR BY POSITION BUCKET
# ---------------------------------------------------------

position_ctr_median = (
    baseline.loc[
        baseline["has_position"],
        ["position_bucket", "observed_ctr"]
    ]
    .groupby(
        "position_bucket",
        observed=False
    )["observed_ctr"]
    .median()
)

print("Median CTR by position bucket:")
display(position_ctr_median)

# ---------------------------------------------------------
# MAP MEDIAN CTR BACK TO EACH ROW
# ---------------------------------------------------------

# Convert the categorical bucket to string before mapping.
baseline["position_bucket_name"] = (
    baseline["position_bucket"].astype(str)
)

median_lookup = position_ctr_median.to_dict()

baseline["position_bucket_median_ctr"] = (
    baseline["position_bucket_name"]
    .map(median_lookup)
)

# Force mapped values to numeric
baseline["position_bucket_median_ctr"] = pd.to_numeric(
    baseline["position_bucket_median_ctr"],
    errors="coerce"
)

# ---------------------------------------------------------
# CALCULATE CTR GAP
# ---------------------------------------------------------

baseline["ctr_gap"] = (
    baseline["position_bucket_median_ctr"]
    - baseline["observed_ctr"]
)

# ---------------------------------------------------------
# SCORE
# ---------------------------------------------------------

baseline["score"] = (
    baseline["has_position"] &
    (baseline["ctr_gap"] > 0)
).astype(int)

# ---------------------------------------------------------
# REASON CODE
# ---------------------------------------------------------

baseline["reason_code"] = np.where(
    baseline["score"] == 1,
    "CTR_FIX",
    "NO_PRIORITY_SIGNAL"
)

# ---------------------------------------------------------
# ACTION
# ---------------------------------------------------------

baseline["action"] = np.where(
    baseline["score"] == 1,
    "CTR_FIX",
    "NO_ACTION"
)

# ---------------------------------------------------------
# RANK
# ---------------------------------------------------------

ranked = (
    baseline
    .sort_values(
        [
            "score",
            "ctr_gap",
            "impressions_90d"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

ranked["rank"] = np.arange(1, len(ranked) + 1)

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print("\nAction counts:")
print(ranked["action"].value_counts())

print("\nReason-code counts:")
print(ranked["reason_code"].value_counts())

print("\nTop 20:")
display(
    ranked[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "observed_ctr",
            "position_bucket_name",
            "position_bucket_median_ctr",
            "ctr_gap",
            "avg_position",
            "impressions_90d",
            "clicks_90d"
        ]
    ].head(20)
)

Low-CTR threshold: 0.0
Rows with measurable position: 28795
Median CTR by position bucket:


position_bucket
Positions 1-5     0.18
Positions 6-10    0.14
Positions 11+     0.04
Name: observed_ctr, dtype: float64


Action counts:
action
NO_ACTION    15809
CTR_FIX      14191
Name: count, dtype: int64

Reason-code counts:
reason_code
NO_PRIORITY_SIGNAL    15809
CTR_FIX               14191
Name: count, dtype: int64

Top 20:


,rank,content_id,score,reason_code,action,observed_ctr,position_bucket_name,position_bucket_median_ctr,ctr_gap,avg_position,impressions_90d,clicks_90d
0,1,content_c82bc0c24241,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,4.3,13676,0
1,2,content_d6e1bbb4a996,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,3.9,4955,0
2,3,content_50dfd64f9e8e,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,2.4,4446,0
3,4,content_134631e65b9e,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,1.5,4417,0
4,5,content_823ea9b9b355,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,3.9,4369,0
5,6,content_ce8619672faf,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,3.5,3855,0
6,7,content_747870c01e28,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,3.2,3585,0
7,8,content_0ad759bc5d3d,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,4.3,3409,0
8,9,content_6c7f71ab52fe,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,4.6,3267,0
9,10,content_cab2a611a390,1,CTR_FIX,CTR_FIX,0.0,Positions 1-5,0.18,0.18,3.4,3160,0


In [5]:
import os

REPO_PATH = os.getcwd() if os.path.exists("work") else "/content/flyrank-ml-internship-starter"

print("Repository exists:", os.path.exists(REPO_PATH))
print("Notebook folder exists:", os.path.exists(f"{REPO_PATH}/work/notebooks"))
print("Output folder exists:", os.path.exists(f"{REPO_PATH}/work/outputs"))

print("\nRepository contents:")
print(os.listdir(REPO_PATH))

Repository exists: True
Notebook folder exists: True
Output folder exists: False

Repository contents:
['.git', '.github', '.gitignore', '.venv', 'AGENTS.md', 'CLAUDE.md', 'data', 'DATA_USE.md', 'docs', 'flyrank-ml-internship-starter', 'GUIDE.md', 'high_priority_refresh_queue.csv', 'LICENSE', 'notebooks', 'outputs', 'README.md', 'refresh_priority_queue.csv', 'requirements.txt', 'scratch', 'scripts', 'SETUP.md', 'skills', 'submission', 'work']


In [6]:
# ---------------------------------------------------------
# WRITE REQUIRED OUTPUT
# ---------------------------------------------------------

REPO_PATH = os.getcwd() if os.path.exists("work") else "/content/flyrank-ml-internship-starter"

OUTPUT_DIR = os.path.join(
    REPO_PATH,
    "work",
    "outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "baseline_action_score.csv"
)

ranked_output = ranked[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "observed_ctr",
        "avg_position",
        "impressions_90d",
        "clicks_90d"
    ]
].copy()

ranked_output.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Ranked queue written successfully.")
print("Output:", OUTPUT_PATH)
print("Rows:", len(ranked_output))

display(ranked_output.head(20))

Ranked queue written successfully.
Output: C:\Users\alina\Desktop\flyrank\work\outputs\baseline_action_score.csv
Rows: 30000


,rank,content_id,score,reason_code,action,observed_ctr,avg_position,impressions_90d,clicks_90d
0,1,content_c82bc0c24241,1,CTR_FIX,CTR_FIX,0.0,4.3,13676,0
1,2,content_d6e1bbb4a996,1,CTR_FIX,CTR_FIX,0.0,3.9,4955,0
2,3,content_50dfd64f9e8e,1,CTR_FIX,CTR_FIX,0.0,2.4,4446,0
3,4,content_134631e65b9e,1,CTR_FIX,CTR_FIX,0.0,1.5,4417,0
4,5,content_823ea9b9b355,1,CTR_FIX,CTR_FIX,0.0,3.9,4369,0
5,6,content_ce8619672faf,1,CTR_FIX,CTR_FIX,0.0,3.5,3855,0
6,7,content_747870c01e28,1,CTR_FIX,CTR_FIX,0.0,3.2,3585,0
7,8,content_0ad759bc5d3d,1,CTR_FIX,CTR_FIX,0.0,4.3,3409,0
8,9,content_6c7f71ab52fe,1,CTR_FIX,CTR_FIX,0.0,4.6,3267,0
9,10,content_cab2a611a390,1,CTR_FIX,CTR_FIX,0.0,3.4,3160,0


In [7]:
# ---------------------------------------------------------
# VERIFY OUTPUT
# ---------------------------------------------------------

assert os.path.exists(OUTPUT_PATH), "CSV was not created."

check_df = pd.read_csv(OUTPUT_PATH)

print("CSV exists:", True)
print("CSV shape:", check_df.shape)
print("CSV columns:")
print(check_df.columns.tolist())

display(check_df.head(10))

CSV exists: True
CSV shape: (30000, 9)
CSV columns:
['rank', 'content_id', 'score', 'reason_code', 'action', 'observed_ctr', 'avg_position', 'impressions_90d', 'clicks_90d']


,rank,content_id,score,reason_code,action,observed_ctr,avg_position,impressions_90d,clicks_90d
0,1,content_c82bc0c24241,1,CTR_FIX,CTR_FIX,0.0,4.3,13676,0
1,2,content_d6e1bbb4a996,1,CTR_FIX,CTR_FIX,0.0,3.9,4955,0
2,3,content_50dfd64f9e8e,1,CTR_FIX,CTR_FIX,0.0,2.4,4446,0
3,4,content_134631e65b9e,1,CTR_FIX,CTR_FIX,0.0,1.5,4417,0
4,5,content_823ea9b9b355,1,CTR_FIX,CTR_FIX,0.0,3.9,4369,0
5,6,content_ce8619672faf,1,CTR_FIX,CTR_FIX,0.0,3.5,3855,0
6,7,content_747870c01e28,1,CTR_FIX,CTR_FIX,0.0,3.2,3585,0
7,8,content_0ad759bc5d3d,1,CTR_FIX,CTR_FIX,0.0,4.3,3409,0
8,9,content_6c7f71ab52fe,1,CTR_FIX,CTR_FIX,0.0,4.6,3267,0
9,10,content_cab2a611a390,1,CTR_FIX,CTR_FIX,0.0,3.4,3160,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

I reviewed the top 20 ranked items as decision-support recommendations rather than guaranteed actions.

For each item, I record the recommended action, why the item was selected, and what could make the recommendation wrong.

The review is intentionally skeptical. A low CTR relative to other content in the same position bucket is a directional signal, not proof that changing search-result presentation will improve performance.

In [8]:
# ---------------------------------------------------------
# TOP-20 REVIEW
# ---------------------------------------------------------

top20 = ranked.head(20).copy()

def why_selected(row):
    if row["reason_code"] == "CTR_FIX":
        return (
            f"CTR ({row['observed_ctr']:.4f}%) is below the "
            f"median CTR for its position bucket "
            f"({row['position_bucket_median_ctr']:.4f}%). "
            "This makes it a directional CTR-fix candidate."
        )
    else:
        return (
            "The item does not meet the baseline's priority condition."
        )


def what_would_make_it_wrong(row):
    if row["reason_code"] == "CTR_FIX":
        return (
            "The recommendation could be wrong if the lower CTR is caused "
            "by search intent, query mix, ranking position, or another "
            "factor that changing the title/snippet would not address."
        )
    else:
        return (
            "The simple baseline may miss an opportunity because it uses "
            "only a limited set of observed signals."
        )


top20_review = top20[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "observed_ctr",
        "avg_position",
        "position_bucket_median_ctr",
        "ctr_gap",
        "impressions_90d",
        "clicks_90d"
    ]
].copy()

top20_review["why_it_is_there"] = top20.apply(
    why_selected,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

display(top20_review)

,rank,content_id,score,reason_code,action,observed_ctr,avg_position,position_bucket_median_ctr,ctr_gap,impressions_90d,clicks_90d,why_it_is_there,what_would_make_it_wrong
0,1,content_c82bc0c24241,1,CTR_FIX,CTR_FIX,0.0,4.3,0.18,0.18,13676,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
1,2,content_d6e1bbb4a996,1,CTR_FIX,CTR_FIX,0.0,3.9,0.18,0.18,4955,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
2,3,content_50dfd64f9e8e,1,CTR_FIX,CTR_FIX,0.0,2.4,0.18,0.18,4446,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
3,4,content_134631e65b9e,1,CTR_FIX,CTR_FIX,0.0,1.5,0.18,0.18,4417,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
4,5,content_823ea9b9b355,1,CTR_FIX,CTR_FIX,0.0,3.9,0.18,0.18,4369,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
5,6,content_ce8619672faf,1,CTR_FIX,CTR_FIX,0.0,3.5,0.18,0.18,3855,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
6,7,content_747870c01e28,1,CTR_FIX,CTR_FIX,0.0,3.2,0.18,0.18,3585,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
7,8,content_0ad759bc5d3d,1,CTR_FIX,CTR_FIX,0.0,4.3,0.18,0.18,3409,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
8,9,content_6c7f71ab52fe,1,CTR_FIX,CTR_FIX,0.0,4.6,0.18,0.18,3267,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...
9,10,content_cab2a611a390,1,CTR_FIX,CTR_FIX,0.0,3.4,0.18,0.18,3160,0,CTR (0.0000%) is below the median CTR for its ...,The recommendation could be wrong if the lower...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline is intentionally simple, so some highly ranked items may be weaker recommendations than their score suggests.

A CTR gap does not prove that a CTR intervention will work. The observed difference may be explained by search intent, query mix, ranking position, SERP features, or other factors not represented in this baseline.

### Leakage check

The baseline does not use FlyRank's existing product flags or label-derived fields as scoring inputs.

It also does not use future-window outcomes. The rule uses only observed historical CTR and average position.

In [9]:
# ---------------------------------------------------------
# WEAK PICKS
# ---------------------------------------------------------

print("Potential weak picks for skeptical review:\n")

# Select a few high-priority rows that have relatively small CTR gaps.
weak_candidates = (
    ranked[
        (ranked["score"] == 1) &
        (ranked["ctr_gap"].notna())
    ]
    .sort_values(
        ["ctr_gap", "impressions_90d"],
        ascending=[True, False]
    )
    .head(5)
)

display(
    weak_candidates[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "observed_ctr",
            "avg_position",
            "position_bucket_median_ctr",
            "ctr_gap",
            "impressions_90d",
            "clicks_90d"
        ]
    ]
)

print(
    "\nThese are potential weak picks because their CTR advantage "
    "over the position-bucket baseline is relatively small."
)

Potential weak picks for skeptical review:



,rank,content_id,score,reason_code,action,observed_ctr,avg_position,position_bucket_median_ctr,ctr_gap,impressions_90d,clicks_90d
14164,14165,content_2ab44dbca810,1,CTR_FIX,CTR_FIX,0.17,4.6,0.18,0.01,30662,53
14165,14166,content_3fc76c6f83bf,1,CTR_FIX,CTR_FIX,0.17,3.5,0.18,0.01,29869,51
14166,14167,content_3a3391080e53,1,CTR_FIX,CTR_FIX,0.17,4.5,0.18,0.01,28404,48
14167,14168,content_604f1e836e0b,1,CTR_FIX,CTR_FIX,0.17,5.0,0.18,0.01,27529,47
14168,14169,content_d76fae202e5c,1,CTR_FIX,CTR_FIX,0.17,3.9,0.18,0.01,22126,37



These are potential weak picks because their CTR advantage over the position-bucket baseline is relatively small.


In [10]:
# ---------------------------------------------------------
# LEAKAGE CHECK
# ---------------------------------------------------------

# Product flags and label-derived fields that must NOT
# be used as inputs to the baseline rule.
forbidden_features = {
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate",
    "trend_direction",
    "trend_pct"
}

# Fields actually used to calculate the score.
rule_inputs = {
    "avg_position",
    "observed_ctr"
}

leaked_inputs = forbidden_features.intersection(rule_inputs)

print("Leakage check")
print("-------------")
print("Rule inputs:", sorted(rule_inputs))
print("Forbidden inputs detected:", leaked_inputs)

assert len(leaked_inputs) == 0

print("\nPASS — no product flags or label-derived fields are used.")

Leakage check
-------------
Rule inputs: ['avg_position', 'observed_ctr']
Forbidden inputs detected: set()

PASS — no product flags or label-derived fields are used.


In [11]:
# ---------------------------------------------------------
# FUTURE-WINDOW CHECK
# ---------------------------------------------------------

future_like_fields = [
    col for col in ranked.columns
    if any(
        term in col.lower()
        for term in [
            "future",
            "next_30",
            "next_60",
            "next_90"
        ]
    )
]

print("Future-like fields detected in ranked dataframe:")
print(future_like_fields)

assert len(future_like_fields) == 0

print("PASS — no future-window fields are used.")

Future-like fields detected in ranked dataframe:
[]
PASS — no future-window fields are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.